# 🚀 AuraFit - Sanal Kabin Sunucusu (Gradio Client)

Bu notebook, **Gradio Client** kullanarak HuggingFace'teki ücretsiz **IDM-VTON** Space'ine istek atar.
- ❌ 8GB model indirmeye gerek yok
- ❌ Token/401 hatası yok
- ❌ GPU bellek sorunu yok
- ✅ Sadece birkaç satır kodla çalışır

### 🛠️ KULLANIM ADIMLARI:
1. Aşağıdaki hücreleri sırasıyla çalıştırın.
2. En alttaki hücrede size özel bir **LocalTunnel bağlantı linki** verilecektir.
3. Bu linki kopyalayıp AuraFit backend projenizdeki `.env` dosyasına `CUSTOM_VTON_API_URL` olarak yapıştırın.

## 📦 1. Gerekli Kütüphanelerin Kurulumu

In [ ]:
!pip install -q gradio_client fastapi uvicorn python-multipart Pillow nest-asyncio
print('✅ Tüm kütüphaneler kuruldu!')

## 🖥️ 2. Sunucu Kodunu Oluşturma

In [ ]:
code = '''import io
import os
import shutil
import tempfile
from PIL import Image
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse
from gradio_client import Client, handle_file

app = FastAPI(title="AuraFit VTON API (Gradio Client)")

# Connect to free IDM-VTON Space
print("[INFO] IDM-VTON Space\'e bağlanılıyor...")
vton_client = Client("yisol/IDM-VTON")
print("[INFO] Bağlantı başarılı! Sunucu hazır.")

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/tryon")
async def tryon(user_image: UploadFile = File(...), product_image: UploadFile = File(...), prompt: str = Form("")):
    user_path = "/tmp/vton_user.jpg"
    product_path = "/tmp/vton_product.jpg"
    
    with open(user_path, "wb") as buffer:
        shutil.copyfileobj(user_image.file, buffer)
    with open(product_path, "wb") as buffer:
        shutil.copyfileobj(product_image.file, buffer)
    
    print("[INFO] VTON isteği alındı, IDM-VTON Space\'e gönderiliyor...")
    
    try:
        result = vton_client.predict(
            dict={"background": handle_file(user_path), "layers": [], "composite": None},
            garm_img=handle_file(product_path),
            garment_des=prompt if prompt else "A garment",
            is_checked=True,
            is_checked_crop=False,
            denoise_steps=30,
            seed=42,
            api_name="/tryon"
        )
        
        # result is a tuple, first element is the output image path
        result_path = result[0] if isinstance(result, (list, tuple)) else result
        print(f"[SUCCESS] VTON tamamlandı: {result_path}")
        
        output_path = "/tmp/vton_result.jpg"
        img = Image.open(result_path).convert("RGB")
        img.save(output_path, "JPEG", quality=95)
        return FileResponse(output_path, media_type="image/jpeg")
        
    except Exception as e:
        print(f"[ERROR] IDM-VTON hatası: {str(e)}")
        # Fallback: basit blending
        try:
            u_img = Image.open(user_path).convert("RGBA")
            p_img = Image.open(product_path).convert("RGBA")
            datas = p_img.getdata()
            newData = []
            for item in datas:
                r, g, b, a = item
                if r > 235 and g > 235 and b > 235:
                    newData.append((255, 255, 255, 0))
                else:
                    newData.append(item)
            p_img.putdata(newData)
            u_w, u_h = u_img.size
            g_w = int(u_w * 0.9)
            ar = p_img.height / p_img.width
            g_h = int(g_w * ar)
            p_r = p_img.resize((g_w, g_h), Image.Resampling.LANCZOS)
            overlay = Image.new("RGBA", u_img.size, (0,0,0,0))
            px = int((u_w - g_w) / 2)
            py = int(u_h * 0.22)
            overlay.paste(p_r, (px, py), p_r)
            composite = Image.alpha_composite(u_img, overlay)
            output_path = "/tmp/vton_result.jpg"
            composite.convert("RGB").save(output_path, "JPEG", quality=95)
            print("[FALLBACK] Basit blending uygulandı.")
            return FileResponse(output_path, media_type="image/jpeg")
        except Exception as fe:
            return FileResponse(user_path, media_type="image/jpeg")
'''

with open('server_app.py', 'w') as f:
    f.write(code)
print('✅ server_app.py oluşturuldu!')

## 🌐 3. API'yi Başlatma ve Canlıya Alma

In [ ]:
import subprocess
import time
import os

print('🚀 Sunucu başlatılıyor...')
subprocess.Popen(['uvicorn', 'server_app:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(8)

!npm install -g localtunnel
print('⚡ LocalTunnel bağlantısı kuruluyor...')
os.system('nohup lt --port 8000 > localtunnel.log 2>&1 &')
time.sleep(5)

print('\n👉 BAĞLANTI LİNKİNİZ:\n')
try:
    with open('localtunnel.log', 'r') as f:
        print(f.read())
except:
    print('Log dosyası henüz oluşmadı, birkaç saniye bekleyin.')

## ✅ 4. Test

In [ ]:
!curl -s -o /dev/null -w "%{http_code}" http://localhost:8000/health